# Paper 3 — 01 · Refusal-direction probe

Builds the refusal-direction probe (METHOD_DESIGN §2) for one anchor model.
One-shot, ~10 GPU-minutes, no API cost. Probe artefacts feed every RD-DPO
training run downstream. Probe set is pre-registered: 100 toxic + 100 benign
Romanian prompts disjoint from train/dev/holdout.

**Inputs:** Paper 2 RoSafetyBench prompts (read from Drive).
**Outputs:** `data/probes/<short>/{block_directions.pt, block_scores.json,
selected_blocks.json, probe_set.jsonl, meta.json, score_curve.png}`.

**Runtime:** ~10 min on A100 for a 3-4B model; ~20 min for 7B.

In [ ]:
%%capture
# Pinned versions match METHOD_DESIGN §4 + requirements.txt. Restart-the-runtime
# is rarely needed because all of these are wheel-only on A100/CUDA 12.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'peft>=0.13' \
    'trl>=0.11' \
    'datasets>=3.0' \
    'bitsandbytes>=0.44' \
    huggingface_hub ipywidgets pyyaml -q
# flash-attn is a separate compile -- best installed once, isolated, and
# allowed to fall back to PyTorch SDPA if the wheel is unavailable for the
# current CUDA/torch combo.
!pip install -U 'flash-attn>=2.6' --no-build-isolation -q || \
    echo '(flash-attn install failed -- will fall back to SDPA at runtime)'

In [ ]:
import os, json, gc, time, hashlib, math, sys
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")

PROBE_DIR    = DRIVE_ROOT / "data" / "probes"
PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
SPLITS_DIR   = DRIVE_ROOT / "data" / "splits"
ADAPTERS_DIR = DRIVE_ROOT / "adapters"
RESULTS_DIR  = DRIVE_ROOT / "results"
LOGS_DIR     = DRIVE_ROOT / "logs"
FIG_DIR      = DRIVE_ROOT / "figures"
for d in [PROBE_DIR, PREFS_DIR, SPLITS_DIR, ADAPTERS_DIR, RESULTS_DIR, LOGS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Paper 2 reuse: judges, llm_judge, holdout splits ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- A100 sanity + perf toggles ---
assert torch.cuda.is_available(), "GPU not available -- switch runtime to A100 GPU."
DEVICE_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {DEVICE_NAME}   VRAM: {VRAM_GB:.1f} GB   torch={torch.__version__}")
if "A100" not in DEVICE_NAME:
    print(f"WARNING: expected A100, got {DEVICE_NAME}. Re-tune BATCH_SIZE/grad-accum below.")

torch.set_float32_matmul_precision("high")          # TF32 for fp32 matmuls
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True               # autotune for fixed shapes

# --- HF / OpenRouter auth (set in Colab Secrets, not in code) ---
try:
    from google.colab import userdata
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
except Exception as _e:
    print(f"(secrets not configured: {_e}; set HF_TOKEN / OPENROUTER_API_KEY in Colab secrets)")


def load_kwargs_for(family: str) -> dict:
    """A100-tuned dtype + attention impl per model family.

    Why bf16 everywhere: A100 has bf16 tensor cores at the same throughput as
    fp16, bf16 has the dynamic range of fp32 (no overflow on long sequences),
    and bf16 is the training dtype for all anchor families. Using fp16 here
    silently broke Gemma 3 in Paper 2 (953 empty-string outputs).
    """
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        # Gemma 3 sliding-window attention is brittle with flash-attn-2.
        return {**common, "attn_implementation": "eager"}
    try:
        import flash_attn  # noqa: F401
        return {**common, "attn_implementation": "flash_attention_2"}
    except ImportError:
        return {**common, "attn_implementation": "sdpa"}


def family_of(anchor_id: str) -> str:
    a = anchor_id.lower()
    if "qwen2.5" in a: return "qwen2.5"
    if "qwen3"   in a: return "qwen3"
    if "llama"   in a: return "llama"
    if "gemma"   in a: return "gemma"
    if "phi"     in a: return "phi"
    return "other"


def short_of(anchor_id: str) -> str:
    return (anchor_id.split("/")[-1].lower()
            .replace("-instruct", "").replace("-it", ""))

print("Bootstrap done.")

## Configuration

`ANCHOR` is the only knob. Re-run once per anchor; cached artefacts are detected at the top of compute cells.

In [ ]:
# Pick one anchor for this run.
# Anchors:    Qwen/Qwen2.5-3B-Instruct, meta-llama/Llama-3.2-3B-Instruct,
#             google/gemma-3-4b-it
# Scaling:    Qwen/Qwen2.5-{0.5B,1.5B,3B,7B}-Instruct
ANCHOR = "Qwen/Qwen2.5-3B-Instruct"

PROBE_SET_SIZE     = 200          # locked at submission; 100 tox + 100 ben
HIDDEN_BATCH_SIZE  = 8            # bf16 + last-position only -> small footprint
K_VALUES           = [1, 2, 4, 8]

short  = short_of(ANCHOR)
family = family_of(ANCHOR)
print(f"ANCHOR : {ANCHOR}")
print(f"family : {family}")
print(f"short  : {short}")

OUT = PROBE_DIR / short
OUT.mkdir(parents=True, exist_ok=True)
print(f"output : {OUT}")

## Probe set (pre-registered, frozen)

Source: RoSafetyBench `toxicity` (harmful intents in Romanian) for the toxic
side, `overrefusal` (benign Romanian prompts that look superficially risky) for
the benign side. First run writes `probe_set.jsonl` to Drive and pins it; later
runs re-use it so layer ranking is comparable across anchors.

In [ ]:
PROBE_SET_PATH = PROBE_DIR / "probe_set.jsonl"

if PROBE_SET_PATH.exists():
    probe_set = [json.loads(l) for l in open(PROBE_SET_PATH, encoding="utf-8")]
    print(f"Loaded frozen probe set ({len(probe_set)} prompts) from {PROBE_SET_PATH}")
else:
    import random
    rng = random.Random(0xC0FFEE)
    tox = [json.loads(l) for l in open(PAPER2_ROOT / "benchmark" / "expanded" / "toxicity.jsonl",   encoding="utf-8")]
    ben = [json.loads(l) for l in open(PAPER2_ROOT / "benchmark" / "expanded" / "overrefusal.jsonl", encoding="utf-8")]
    rng.shuffle(tox); rng.shuffle(ben)
    probe_set = (
        [{"id": x["id"], "text_ro": x["text_ro"], "y": "tox"} for x in tox[:100]] +
        [{"id": x["id"], "text_ro": x["text_ro"], "y": "ben"} for x in ben[:100]]
    )
    rng.shuffle(probe_set)
    with open(PROBE_SET_PATH, "w", encoding="utf-8") as f:
        for x in probe_set: f.write(json.dumps(x, ensure_ascii=False) + "\n")
    print(f"Constructed probe set ({len(probe_set)} prompts) -> {PROBE_SET_PATH}")

SHA   = hashlib.sha256(PROBE_SET_PATH.read_bytes()).hexdigest()[:16]
n_tox = sum(1 for x in probe_set if x["y"] == "tox")
n_ben = sum(1 for x in probe_set if x["y"] == "ben")
print(f"sha256[:16] = {SHA}   tox={n_tox}   ben={n_ben}")

## Capture hidden states

We capture the residual-stream activation at the **last position of the
assistant prefix** (immediately before the first generated token). Single
forward pass per prompt with `output_hidden_states=True`, no generation.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np

OUT_FILE = OUT / "hidden_states.pt"
if OUT_FILE.exists():
    print(f"Cached hidden states found at {OUT_FILE}; skipping forward pass.")
    cache = torch.load(OUT_FILE)
    H, ys = cache["H"], cache["ys"]
else:
    tok = AutoTokenizer.from_pretrained(ANCHOR, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(ANCHOR, **load_kwargs_for(family))
    mdl.eval()

    formatted, ys = [], []
    for x in probe_set:
        text = tok.apply_chat_template(
            [{"role": "user", "content": x["text_ro"]}],
            tokenize=False, add_generation_prompt=True,
        )
        formatted.append(text); ys.append(x["y"])

    with torch.no_grad():
        ids = tok(formatted[:1], return_tensors="pt", padding=True,
                  truncation=True, max_length=1024).to(mdl.device)
        out = mdl(**ids, output_hidden_states=True, use_cache=False)
        n_blocks = len(out.hidden_states); d_model = out.hidden_states[0].shape[-1]
        print(f"n_blocks = {n_blocks}   d_model = {d_model}")

        H = torch.zeros(len(formatted), n_blocks, d_model, dtype=torch.float32)
        t0 = time.time()
        for i in range(0, len(formatted), HIDDEN_BATCH_SIZE):
            batch = formatted[i:i + HIDDEN_BATCH_SIZE]
            ids = tok(batch, return_tensors="pt", padding=True,
                      truncation=True, max_length=1024).to(mdl.device)
            out = mdl(**ids, output_hidden_states=True, use_cache=False)
            last_idx = ids["input_ids"].shape[1] - 1
            for layer_ix, hs in enumerate(out.hidden_states):
                H[i:i + len(batch), layer_ix, :] = hs[:, last_idx, :].float().cpu()
            if (i // HIDDEN_BATCH_SIZE) % 4 == 0:
                rate = (i + len(batch)) / max(time.time() - t0, 1e-6)
                print(f"  {i + len(batch):>4}/{len(formatted)}  {rate:.1f}/s")

    torch.save({"H": H, "ys": ys}, OUT_FILE)
    print(f"Saved hidden states -> {OUT_FILE}")
    del mdl, tok
    gc.collect(); torch.cuda.empty_cache()

## Score and rank blocks

`s_ℓ = ‖μ_tox − μ_ben‖₂ × |cos(d_ℓ, v_ℓ)|` where `d_ℓ` is the difference-in-means direction and `v_ℓ` is the principal component of toxic-class activations. High score = good separation **and** unidirectional toxic-class spread.

In [ ]:
H_np = H.numpy()
y    = np.array([1 if v == "tox" else 0 for v in ys])
n_blocks, d_model = H_np.shape[1], H_np.shape[2]

block_directions = np.zeros((n_blocks, d_model), dtype=np.float32)
block_scores = []
for L in range(n_blocks):
    Ht = H_np[y == 1, L, :]; Hb = H_np[y == 0, L, :]
    mu_t = Ht.mean(axis=0);  mu_b = Hb.mean(axis=0)
    delta = mu_t - mu_b
    sep = float(np.linalg.norm(delta))
    d   = delta / max(sep, 1e-9)
    block_directions[L] = d.astype(np.float32)
    Ht_c = Ht - mu_t
    _, _, Vt = np.linalg.svd(Ht_c, full_matrices=False)
    cos = float(abs(d @ Vt[0]))
    block_scores.append({"layer": L, "separation": sep,
                         "unidirectionality": cos, "score": sep * cos})

ranked = sorted(block_scores, key=lambda r: r["score"], reverse=True)

early_cutoff = max(1, n_blocks // 4)
selected = {}
for k in K_VALUES:
    keep = []
    for r in ranked:
        if r["layer"] < early_cutoff: continue
        keep.append(r["layer"])
        if len(keep) == k: break
    selected[k] = sorted(keep)

top4 = float(np.mean([r["score"] for r in ranked[:4]]))
bot4 = float(np.mean([r["score"] for r in ranked[-4:]] or [1e-9]))
ratio = top4 / max(bot4, 1e-9)
print(f"separation range: {min(r['separation'] for r in block_scores):.3f}"
      f"  -> {max(r['separation'] for r in block_scores):.3f}")
print(f"top-4 / bottom-4 score ratio = {ratio:.2f}  (target >= 2.0)")
print(f"selected blocks: {selected}")

## Save artefacts

All written under `data/probes/<short>/`. Probe set + selected blocks ship with the public release.

In [ ]:
torch.save({"directions": torch.from_numpy(block_directions)}, OUT / "block_directions.pt")
(OUT / "block_scores.json").write_text(json.dumps(block_scores, indent=2))
(OUT / "selected_blocks.json").write_text(json.dumps(selected, indent=2))
(OUT / "meta.json").write_text(json.dumps({
    "anchor": ANCHOR, "short": short, "family": family,
    "n_blocks": n_blocks, "d_model": d_model,
    "probe_set_path": str(PROBE_SET_PATH), "probe_set_sha256_16": SHA,
    "n_tox": n_tox, "n_ben": n_ben,
    "early_cutoff_layer": early_cutoff,
    "score_top_bottom_ratio_4": ratio,
    "torch_version": torch.__version__,
    "device": DEVICE_NAME, "vram_gb": round(VRAM_GB, 1),
    "built_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
}, indent=2))

import matplotlib.pyplot as plt
layers = [r["layer"] for r in block_scores]
scores = [r["score"] for r in block_scores]
plt.figure(figsize=(7, 3))
plt.bar(layers, scores)
for k_, blocks in selected.items():
    for L in blocks:
        plt.bar([L], [scores[L]], color="tab:red")
plt.title(f"refusal-direction score per block — {short}")
plt.xlabel("block"); plt.ylabel("sep × |cos|")
plt.tight_layout()
plt.savefig(OUT / "score_curve.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"\nDone. Artefacts at {OUT}")